# E1 memory feasibility on Kaggle CPU

Set **Accelerator → None** and run this notebook on its own. The default cell is the full authors' NZEnergy stream with Sgb(Arf), seed 1. Only the JVM maximum heap changes to 16 GiB. This tests feasibility for one cell on the measured Kaggle session.

Set `PROJECT` to the project checkout or attached Kaggle Dataset containing both Python scripts in `scripts/`. Supply a staged MOA fat jar built from the pinned authors' fork, its SHA-256, and the path to the matching Java executable if it is outside `PATH`. See `docs/e1_memory_feasibility.md` for the build and measurement details.

With Internet enabled, leave `UPSTREAM_CHECKOUT` unset to fetch the authors' public pinned revision. For an offline run, supply a Git checkout containing that revision. Use a fresh output path for each attempt and allow time to save the notebook outputs.

In [ ]:
from pathlib import Path

PROJECT = Path("/kaggle/working/SGBR-Reproduction")
UPSTREAM_CHECKOUT = None  # Set to a staged Git checkout for offline runs.
MOA_JAR = None  # Path to the trusted fat jar from the pinned fork.
MOA_SHA256 = ""  # SHA-256 recorded at build time.
JAVA = "java"  # Or an absolute path to the matching JDK's bin/java.

DATASET = "NZEnergy"
LEARNER = "SGBT(ARFReg(FIRTDD))"  # Sgb(Arf), using the authors' config key.
SEED = 1
TIMEOUT_SECONDS = None  # Optional: leave time to export results before the session ends.
MANIFEST = Path("/kaggle/working/results/manifest/e1_memory_feasibility.json")


In [ ]:
import json
import re
import subprocess
import sys
from tempfile import TemporaryDirectory

WORKING = Path("/kaggle/working")
UPSTREAM = "https://github.com/nuwangunasekara/SGBR.git"
PINNED_COMMIT = "f62f7b437512e020f57c4bcd03884c4724eda5b1"

if not WORKING.is_dir():
    raise RuntimeError("Use a Kaggle CPU notebook with /kaggle/working available.")
PROJECT = Path(PROJECT).resolve()
required = [PROJECT / "scripts" / name for name in (
    "e1_memory_feasibility.py", "kaggle_environment_smoke.py",
)]
for script in required:
    if not script.is_file():
        raise FileNotFoundError(f"Set PROJECT to the directory containing {script.name} under scripts/.")
if MOA_JAR is None or not Path(MOA_JAR).is_file() or not re.fullmatch(r"[0-9a-fA-F]{64}", MOA_SHA256):
    raise ValueError("Set MOA_JAR to the staged fat jar and MOA_SHA256 to its recorded digest.")
MANIFEST = Path(MANIFEST).resolve()
if not MANIFEST.is_relative_to(WORKING):
    raise ValueError("Save the manifest in /kaggle/working so that Kaggle retains it.")

with TemporaryDirectory(prefix="sgbr-e1-upstream-", dir=WORKING) as temporary:
    source = Path(UPSTREAM_CHECKOUT).resolve() if UPSTREAM_CHECKOUT else Path(temporary)
    if not UPSTREAM_CHECKOUT:
        subprocess.run(["git", "init", "-q", str(source)], check=True)
        subprocess.run(
            ["git", "-C", str(source), "fetch", "--depth=1", UPSTREAM, PINNED_COMMIT],
            check=True, timeout=300,
        )
    command = [
        sys.executable, str(required[0]),
        "--upstream-checkout", str(source), "--moa-jar", str(Path(MOA_JAR).resolve()),
        "--moa-sha256", MOA_SHA256, "--java", JAVA, "--manifest", str(MANIFEST),
        "--dataset", DATASET, "--learner", LEARNER, "--seed", str(SEED),
    ]
    if TIMEOUT_SECONDS is not None:
        command.extend(["--timeout-seconds", str(TIMEOUT_SECONDS)])
    completed = subprocess.run(command, cwd=WORKING, check=False)

if MANIFEST.is_file():
    result = json.loads(MANIFEST.read_text())
    for key in ("status", "processed_instances", "peak_java_rss_bytes", "memory_cgroup_latest",
                "wall_clock_seconds", "exit_code", "termination_signal", "oom"):
        print(f"{key}: {result.get(key)}")
if completed.returncode:
    raise RuntimeError(f"Feasibility run did not complete. Inspect {MANIFEST} and the Java log.")
